In [ ]:
import pandas as pd
import spacy
from tqdm import tqdm
from huggingface_hub import hf_hub_download, snapshot_download
import sys
import json
import random

nlp = spacy.load("en_core_web_md")

ckpt_path = hf_hub_download(
    repo_id="yeomtong/srl_bert_model",
    filename="best_srl_Sep_29.ckpt")
repo_dir = snapshot_download("yeomtong/srl_bert_model")
sys.path.append(repo_dir)

from predictor import srl_init, predict_sentence
from model import PredicateAwareSRL
from visualizer import prediction, prediction_formatted, create_dict

srl_init(ckpt_path, bert_name= "bert-base-cased")



In [ ]:
# train_df = pd.read_csv("/Enter-your-path/english_train_4-14.csv")
# # df.head()
# dev_df = pd.read_csv("/Enter-your-path/english_val_4-14.csv")
test_df = pd.read_csv("/Enter-your-path/english_test_4-14.csv")

In [ ]:
len(dev_df)

In [ ]:
res = prediction_formatted(test_df['Utterance'][0])
res

In [ ]:
import re
import unicodedata

def clean_text_for_srl(text):
    if not isinstance(text, str):
        return ""

    # Normalize Unicode
    text = unicodedata.normalize("NFKC", text)
    text = text.encode("utf-8", "ignore").decode("utf-8")
    text = text.replace("Â", "").replace("Ã", "A").replace("â€”", "—").replace("â€“", "-")
    text = text.replace("±", "+/-").replace("°", " degrees ")
    text = re.sub(r"\([^)]{40,}\)", "", text)   # overly long (...)
    text = re.sub(r"http\S+|www\S+", "", text)  # URLs
    text = re.sub(r"\s+", " ", text).strip()

    # Truncate long sentences (SRL model's limit is 150)
    if len(text.split()) > 150:
        text = " ".join(text.split()[:150])

    return text

In [ ]:
# Given word list and BIO tags for one predicate frame, return [pred_idx, arg0_idx, arg1_idx, arg2_idx, argm_idx] 
def extract_arg_heads(words, tags):
    pred_idx = -1
    arg0_idx = -1
    arg1_idx = -1
    arg2_idx = -1
    argm_idx = -1

    for i, tag in enumerate(tags):
        # predicate
        if tag == "B-V" and pred_idx == -1:
            pred_idx = i
        if tag.startswith("B-ARG0") and arg0_idx == -1:
            arg0_idx = i
        if tag.startswith("B-ARG1") and arg1_idx == -1:
            arg1_idx = i
        if tag.startswith("B-ARG2") and arg2_idx == -1:
            arg2_idx = i
        if tag.startswith("B-ARGM") and argm_idx == -1:
            argm_idx = i

    return [pred_idx, arg0_idx, arg1_idx, arg2_idx, argm_idx]


In [ ]:
# Run SRL once per utterance and return one record per sentence, each with the same politeness label.
def srl_info_for_utterance(utterance_text, politeness_score):
    utterance_text = clean_text_for_srl(utterance_text)
    if not isinstance(utterance_text, str) or not utterance_text.strip():
        return []
    
    # Split utterance into sentences -> process by sentence
    doc = nlp(utterance_text)
    sentences = [sent.text.strip() for sent in doc.sents]
    # print(sentences)
    
    records = {}
    records['srl_frames'] = []
    for sent in sentences:
        if not sent:
            continue
        
        # === Model 1: all verbs (root_only=False) ===
        words, frames = predict_sentence(sent, root_only=False)
        # print(words, frames)
        srl_out = create_dict(words, frames)
        
        # === Model 2: just head verb (root_only=True)
        # === NOTE: prediction_formatted calls predict_sentence and create_dict (funcs used above)
        # srl_out = prediction_formatted(sent)
        
        words = srl_out.get("words", [])
        verbs = srl_out.get("verbs", [])
        
        for frame in verbs:
            tags = frame.get("tags", [])
            if len(tags) != len(words):
                continue

            heads = extract_arg_heads(words, tags)
            pred_idx = heads[0]
            if pred_idx < 0:
                continue

            rec = {
                "words": words,
                "predicate_word_idx": pred_idx,
                "labels": tags,
                "arg_head_idx": heads,
                # "politeness": float(politeness_score), # or sentiment, etc
            }
            records['srl_frames'].append(rec)
            # print(len(records))
    
    records['politeness'] = politeness_score
    
    return records


In [ ]:
from tqdm import tqdm

def get_srl_processed(df):
    all_records = []
    for  _, row in tqdm(df.iterrows(), total=len(df), desc="Building SRL-aware politeness samples"):
        text = row["Utterance"]
        politeness = row["labels"]
        records = srl_info_for_utterance(text, politeness)
        # records['idx'] = idx 
        all_records.append(records)
        
    return all_records

# train_records = get_srl_processed(train_df)
# dev_records = get_srl_processed(dev_df)
test_records = get_srl_processed(test_df)

In [ ]:
base_path = "/Enter-your-path/SRL_processed/"
# NOTE: Adjust for intended model
model_ext = "model_all_SRL"
full_path = base_path

# Save as jsonl
# with open(full_path + "/srl_politeness_train.jsonl", "w") as f:
#     for rec in train_records:
#         f.write(json.dumps(rec) + "\n")

# with open(full_path + "/srl_politeness_dev.jsonl", "w") as f:
#     for rec in dev_records:
#         f.write(json.dumps(rec) + "\n")

with open(full_path + "/test_update/srl_politeness_test.jsonl", "w") as f:
    for rec in test_records:
        f.write(json.dumps(rec) + "\n")

print("Data saved successfully!")

In [ ]:
from typing import List, Dict, Optional
from dataclasses import dataclass, field

@dataclass
class SRLFrame:
    predicate_word_idx: int
    labels: List[str]
    predicate_form: Optional[str] = None
    arg_head_idx: Optional[List[int]] = None

@dataclass
class UtteranceSample:
    words: List[str]
    frames: List[SRLFrame]
    politeness: Optional[float] = None

def convert_example_to_utterance_level(ex: dict, sep_token: str = None) -> UtteranceSample:
    """
    Convert one JSON example:
      {"srl_frames":[{words, predicate_word_idx, labels, ...}, ...], "politeness": ...}
    into:
      UtteranceSample(words=utterance_words, srl_frames=[aligned frames...])

    sep_token:
      - If you want to insert a marker token between sentences, set sep_token="<SEP>".
      - If None, we simply concatenate.
    """
    frames_in = ex["srl_frames"]
    politeness = ex.get("politeness", None)

    # 1) Identify unique sentences by their "words" list (order preserved)
    #    We keep them in the order they appear in frames.
    sentence_list = []
    sent2id = {}
    for fr in frames_in:
        w = tuple(fr["words"])
        if w not in sent2id:
            sent2id[w] = len(sentence_list)
            sentence_list.append(list(fr["words"]))

    # 2) Build utterance_words and sentence offsets
    utter_words = []
    offsets = []  # start idx for each sentence
    for s_words in sentence_list:
        offsets.append(len(utter_words))
        utter_words.extend(s_words)
        if sep_token is not None:
            utter_words.append(sep_token)

    # If sep_token is used, adjust offsets of subsequent sentences accordingly
    # (Already handled by appending sep_token after each sentence.)

    N = len(utter_words)

    # Helper: expand a sentence label sequence into utterance-length labels
    def expand_labels(sent_labels, start, sent_len):
        out = ["O"] * N
        out[start:start + sent_len] = sent_labels
        return out

    aligned_frames = []
    for fr in frames_in:
        sent_words = list(fr["words"])
        sent_key = tuple(sent_words)
        sid = sent2id[sent_key]
        start = offsets[sid]
        sent_len = len(sent_words)

        # If you inserted sep_token, the sentence span is still contiguous,
        # because sep is appended AFTER the sentence.
        pred_idx_utt = fr["predicate_word_idx"] + start

        labels_utt = expand_labels(fr["labels"], start, sent_len)

        # arg_head_idx shifting (only if arg_head_idx stores word indices in sentence coords)
        # Your arg_head_idx example looks like a short list; I can't be 100% sure of semantics.
        # Here is a safe default: shift any non-negative index by start.
        arg_head = fr.get("arg_head_idx", None)
        if arg_head is not None:
            shifted = []
            for x in arg_head:
                if isinstance(x, int) and x >= 0:
                    shifted.append(x + start)
                else:
                    shifted.append(x)
            arg_head = shifted

        aligned_frames.append(
            SRLFrame(
                predicate_word_idx=pred_idx_utt,
                labels=labels_utt,
                arg_head_idx=arg_head,
                # sentence_words=sent_words,   # optional debug
            )
        )
        
    if len(aligned_frames) == 0:
        aligned_frames.append(SRLFrame(
            predicate_word_idx=0,
            labels=["O"] * len(utter_words),
            predicate_form=None,
            arg_head_idx=None
        ))

    return UtteranceSample(words=utter_words, frames=aligned_frames, politeness=politeness)


def convert_jsonl(in_path: str, out_path: str, sep_token: str = None):
    with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            ex = json.loads(line)
            utt = convert_example_to_utterance_level(ex, sep_token=sep_token)
            # Write as dict
            out = {
                "words": utt.words,
                "srl_frames": [
                    {
                        "predicate_word_idx": fr.predicate_word_idx,
                        "labels": fr.labels,
                        "arg_head_idx": fr.arg_head_idx,
                    }
                    for fr in utt.frames
                ],
                "politeness": utt.politeness,
            }
            fout.write(json.dumps(out) + "\n")



In [ ]:
base_path = full_path
# convert_jsonl(
#     base_path + "/srl_politeness_train.jsonl",
#     base_path + "/srl_politeness_train.aligned.jsonl"
# )
# convert_jsonl(
#     base_path + "/srl_politeness_dev.jsonl",
#     base_path + "/srl_politeness_dev.aligned.jsonl"
# )
convert_jsonl(
    base_path + "/test_update/srl_politeness_test.jsonl",
    base_path + "/test_update/srl_politeness_test.aligned.jsonl"
)